[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C15_Classic_Architectures_Course/03_rnn_from_scratch/03_rnn_from_scratch.ipynb)

# 03 · RNN 与 BPTT（纯 numpy 从零）

目标：从零实现 **vanilla RNN 的前向与 BPTT 反向**，用**数值梯度检验**焊死正确性；再用 `np.linalg.eigvals` 量化**梯度消失/爆炸**与谱半径的关系；最后实现**梯度裁剪**。

路线：RNN cell 前向 → 整段序列前向 → BPTT 反向(对每个梯度做数值检验) → 梯度消失/爆炸(谱半径) → 梯度裁剪 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(字符级序列)。

> 核心纪律：**权重共享 ⇒ 梯度求和**（dWx/dWh/db 在所有时间步累加），且每个反向梯度都要过数值梯度检验（相对误差 < 1e-5）。

## 0 · 工具：数值梯度检验（全课金标准）

反向传播没有现成参考，用**中心差分**当 ground truth。这是本模块所有 backward 正确性的唯一硬证据。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def numerical_grad(f, x, eps=1e-5):
    '''中心差分逐元素估计 df/dx。f: ndarray->标量。会原地扰动 x 再复原。'''
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f(x)
        x[idx] = old - eps; fm = f(x)
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_error(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.max(np.abs(a - b) / (np.maximum(1e-8, np.abs(a) + np.abs(b))))

# 自检：f(x)=sum(x^2) -> grad 2x
x = rng.standard_normal((3, 4))
assert rel_error(2 * x, numerical_grad(lambda x: np.sum(x ** 2), x.copy())) < 1e-7
print('✅ 数值梯度工具就绪')

## 1 · RNN cell 前向：一步更新

单步：$a_t = W_x x_t + W_h h_{t-1} + b,\ h_t = \tanh(a_t)$。

三个权重 `Wx,Wh,b` 是**唯一**参数，所有时间步共享。下面实现单步并检查形状。

In [ ]:
def rnn_cell_forward(x_t, h_prev, Wx, Wh, b):
    '''单步：返回 (h_t, cache)。x_t:(D,), h_prev:(H,)。'''
    a = Wx @ x_t + Wh @ h_prev + b      # (H,)
    h = np.tanh(a)
    cache = (x_t, h_prev, h)            # 反向要用
    return h, cache

D, H = 3, 4
Wx = rng.standard_normal((H, D)) * 0.5
Wh = rng.standard_normal((H, H)) * 0.5
b  = rng.standard_normal(H) * 0.1
x_t = rng.standard_normal(D)
h_prev = np.zeros(H)
h, _ = rnn_cell_forward(x_t, h_prev, Wx, Wh, b)
print('h_t =', np.round(h, 4))
assert h.shape == (H,)
assert np.all(np.abs(h) < 1.0), 'tanh 输出必在 (-1,1)'
print('✅ RNN cell 单步前向：形状对、tanh 把状态压在 (-1,1)')

## 2 · 整段序列前向：把 cell 沿时间展开

给定输入序列 `xs`（形状 `(T, D)`）与初始 `h0`，逐步调用 cell，得到所有隐藏态 `(T, H)`。

这正是 HTML 里「时间展开」的代码版：一个 for 循环，**同一套权重**用 T 次。

In [ ]:
def rnn_forward(xs, h0, Wx, Wh, b):
    '''xs:(T,D), h0:(H,) -> H_all:(T,H), caches(list).'''
    T = xs.shape[0]; H = h0.shape[0]
    H_all = np.zeros((T, H))
    caches = []
    h_prev = h0
    for t in range(T):
        h, cache = rnn_cell_forward(xs[t], h_prev, Wx, Wh, b)
        H_all[t] = h
        caches.append(cache)
        h_prev = h                      # 本步输出 = 下步输入
    return H_all, caches

T = 5
xs = rng.standard_normal((T, D))
h0 = np.zeros(H)
H_all, caches = rnn_forward(xs, h0, Wx, Wh, b)
print('H_all shape =', H_all.shape, '(每行是一个时间步的隐藏态)')
assert H_all.shape == (T, H) and len(caches) == T
print('✅ 整段前向：T 个时间步共享同一套 Wx,Wh,b')

## 3 · BPTT 反向：核心！权重共享 ⇒ 梯度求和

总损失取 $L = \tfrac12\sum_t \|h_t\|^2$（一个能给**每个时间步**都灌入梯度的简单标量，便于检验）。

反向从 $t=T$ 到 $1$：每步的隐藏态梯度 = **本步直接损失** + **来自未来的** $W_h^\top da_{t+1}$；穿过 tanh 得 $da_t$；三个权重梯度 **累加**（`+=`）。

**头号陷阱**：`dWx/dWh/db` 必须在循环外清零、循环内累加；漏掉累加 = 数值检验必挂。

In [ ]:
def rnn_loss_and_grads(xs, h0, Wx, Wh, b):
    '''前向 + BPTT 反向。损失 L = 0.5 * sum(h_t^2)。
       返回 L, 梯度 dWx,dWh,db,dxs,dh0。'''
    H_all, caches = rnn_forward(xs, h0, Wx, Wh, b)
    T = xs.shape[0]; H = h0.shape[0]
    L = 0.5 * np.sum(H_all ** 2)
    # 反向：先清零（累加的容器）
    dWx = np.zeros_like(Wx); dWh = np.zeros_like(Wh); db = np.zeros_like(b)
    dxs = np.zeros_like(xs)
    dh_next = np.zeros(H)                      # 来自未来的梯度，t=T 时为 0
    for t in reversed(range(T)):
        x_t, h_prev, h = caches[t]
        dh = H_all[t] + dh_next                # dL/dh_t = 本步(0.5h^2->h) + 未来
        da = dh * (1 - h ** 2)                 # 穿过 tanh: tanh'(a)=1-h^2
        dWx += np.outer(da, x_t)               # ★ 累加！权重共享
        dWh += np.outer(da, h_prev)            # ★ 累加！
        db  += da                             # ★ 累加！
        dxs[t] = Wx.T @ da                     # 传给输入
        dh_next = Wh.T @ da                    # 传给上一步隐藏态
    dh0 = dh_next
    return L, dWx, dWh, db, dxs, dh0

L, dWx, dWh, db, dxs, dh0 = rnn_loss_and_grads(xs, h0, Wx, Wh, b)
print(f'L = {L:.4f}')
print('dWh shape', dWh.shape, '| dWx shape', dWx.shape)
assert dWx.shape == Wx.shape and dWh.shape == Wh.shape and db.shape == b.shape
print('✅ BPTT 反向跑通（形状对）；下一格用数值梯度验证数值对')

### 3.1 数值梯度检验：每个梯度都必须过关

对 `Wx, Wh, b, xs, h0` 五者分别用中心差分检验。**这是「我真的推对了」的唯一硬证据。**

In [ ]:
def L_only(xs_, h0_, Wx_, Wh_, b_):
    return rnn_loss_and_grads(xs_, h0_, Wx_, Wh_, b_)[0]

eWx = rel_error(dWx, numerical_grad(lambda W: L_only(xs, h0, W, Wh, b), Wx.copy()))
eWh = rel_error(dWh, numerical_grad(lambda W: L_only(xs, h0, Wx, W, b), Wh.copy()))
eb  = rel_error(db,  numerical_grad(lambda B: L_only(xs, h0, Wx, Wh, B), b.copy()))
exs = rel_error(dxs, numerical_grad(lambda X: L_only(X, h0, Wx, Wh, b), xs.copy()))
eh0 = rel_error(dh0, numerical_grad(lambda H0: L_only(xs, H0, Wx, Wh, b), h0.copy()))
for name, e in [('dWx', eWx), ('dWh', eWh), ('db', eb), ('dxs', exs), ('dh0', eh0)]:
    print(f'{name:5s} 相对误差 = {e:.2e}')
    assert e < 1e-5, f'{name} 反向不正确！'
print('✅✅ 全部 BPTT 梯度通过数值检验 (rel_error < 1e-5) —— 权重共享的求和写对了')

## 4 · 梯度消失 / 爆炸：谱半径说了算

把循环简化成**线性**递推 $h_t = W h_{t-1}$（去掉 tanh，看纯矩阵连乘）。

梯度从第 $T$ 步传回第 0 步要乘 $T$ 次 $W^\top$，范数 $\sim \rho^T$（$\rho$=谱半径）。$\rho<1$ → 指数衰减(消失)；$\rho>1$ → 指数增长(爆炸)。用 `np.linalg.eigvals` 算 $\rho$ 并实测验证。

In [ ]:
def make_W_with_spectral_radius(H, target_rho, seed):
    '''造一个谱半径 = target_rho 的矩阵。取对称矩阵 -> 特征值全为实数，
       幂迭代的每步比值会单调收敛到谱半径，便于干净地验证理论。'''
    g = np.random.default_rng(seed)
    A = g.standard_normal((H, H))
    W = (A + A.T) / 2                          # 对称 => 实特征值
    rho0 = np.max(np.abs(np.linalg.eigvals(W)))
    return W * (target_rho / rho0)            # 缩放到指定谱半径

def grad_norms_through_time(W, n_steps):
    '''模拟梯度沿时间反传：每步乘 W^T，记录范数。'''
    H = W.shape[0]
    g = np.random.default_rng(1).standard_normal(H)
    g /= np.linalg.norm(g)
    norms = []
    for _ in range(n_steps):
        g = W.T @ g
        norms.append(np.linalg.norm(g))
    return np.array(norms)

Hbig = 8; n = 30                            # 用 Hbig，别覆盖前面 RNN 的 H=4
W_decay   = make_W_with_spectral_radius(Hbig, 0.5, seed=2)
W_explode = make_W_with_spectral_radius(Hbig, 1.5, seed=2)
rho_d = np.max(np.abs(np.linalg.eigvals(W_decay)))
rho_e = np.max(np.abs(np.linalg.eigvals(W_explode)))
nd = grad_norms_through_time(W_decay, n)
ne = grad_norms_through_time(W_explode, n)
print(f'谱半径 0.5 : 梯度范数 {nd[0]:.3e} -> {nd[-1]:.3e}  (消失)')
print(f'谱半径 1.5 : 梯度范数 {ne[0]:.3e} -> {ne[-1]:.3e}  (爆炸)')
assert nd[-1] < nd[0] * 1e-3, '谱半径<1 应指数衰减'
assert ne[-1] > ne[0] * 1e3,  '谱半径>1 应指数增长'
# 后期每步比值应收敛到谱半径（迭代对齐主特征向量）
ratio_d = nd[-1] / nd[-2]; ratio_e = ne[-1] / ne[-2]
print(f'后期每步比值: 衰减 {ratio_d:.3f} vs rho {rho_d:.3f} | 爆炸 {ratio_e:.3f} vs rho {rho_e:.3f}')
assert abs(ratio_d - rho_d) < 0.05 and abs(ratio_e - rho_e) < 0.05
print('✅ 实测每步比值收敛到谱半径 —— 消失/爆炸是连乘的数学必然')

## 5 · 梯度裁剪：治爆炸的标准手段

按范数裁剪：$g \leftarrow g\cdot\min(1, \tau/\|g\|)$。两个性质：**方向不变**（等比缩放）、**只在超阈值时生效**。

把多个参数梯度当一个整体（拼接后算总范数）裁剪——这才是真实训练里的做法。

In [ ]:
def clip_grads_by_norm(grads, max_norm):
    '''grads: list of ndarray。整体按 L2 范数裁剪，返回裁剪后的 list 与原范数。'''
    total_sq = sum(np.sum(g ** 2) for g in grads)
    total_norm = np.sqrt(total_sq)
    scale = min(1.0, max_norm / (total_norm + 1e-12))
    return [g * scale for g in grads], total_norm

# 造一个范数很大的梯度（模拟爆炸）
big = [rng.standard_normal((4, 4)) * 10, rng.standard_normal(4) * 10]
clipped, orig_norm = clip_grads_by_norm(big, max_norm=5.0)
new_norm = np.sqrt(sum(np.sum(g ** 2) for g in clipped))
print(f'原范数 = {orig_norm:.2f} -> 裁剪后 = {new_norm:.2f} (阈值 5.0)')
assert new_norm <= 5.0 + 1e-6, '裁剪后范数应 <= 阈值'
# 方向不变：裁剪前后向量平行（余弦 = 1）
flat_b = np.concatenate([g.ravel() for g in big])
flat_c = np.concatenate([g.ravel() for g in clipped])
cos = flat_b @ flat_c / (np.linalg.norm(flat_b) * np.linalg.norm(flat_c))
print(f'裁剪前后方向余弦 = {cos:.6f}')
assert abs(cos - 1.0) < 1e-6, '裁剪应保持方向'
# 小范数梯度不被改动
small = [rng.standard_normal(4) * 0.1]
cl2, _ = clip_grads_by_norm(small, max_norm=5.0)
assert np.allclose(cl2[0], small[0]), '范数 < 阈值时应原样通过'
print('✅ 梯度裁剪：超阈值则等比缩放(方向不变)，未超则原样通过')

## 6 · 串起来：一步带裁剪的 BPTT 更新

把前向、BPTT、裁剪、参数更新拼成**一个训练步**，验证「损失确实下降」——确认整条链路自洽。

In [ ]:
# 一个能学的玩具任务：让末态隐藏均值逼近目标 0.5（回归）
def loss_and_grads_task(xs, h0, Wx, Wh, b, target=0.5):
    H_all, caches = rnn_forward(xs, h0, Wx, Wh, b)
    pred = H_all[-1].mean()                      # 用末态均值当预测
    L = 0.5 * (pred - target) ** 2
    T, H = xs.shape[0], h0.shape[0]
    dWx = np.zeros_like(Wx); dWh = np.zeros_like(Wh); db = np.zeros_like(b)
    dh_next = np.zeros(H)
    dpred = (pred - target)
    for t in reversed(range(T)):
        x_t, h_prev, h = caches[t]
        dh = dh_next.copy()
        if t == T - 1:
            dh = dh + dpred / H                  # 只有末态进入预测
        da = dh * (1 - h ** 2)
        dWx += np.outer(da, x_t); dWh += np.outer(da, h_prev); db += da
        dh_next = Wh.T @ da
    return L, [dWx, dWh, db]

Ht, Dt = 8, 3                               # 这个小实验自带维度，不依赖前面的 H/D
Wx2 = rng.standard_normal((Ht, Dt)) * 0.3
Wh2 = make_W_with_spectral_radius(Ht, 0.9, seed=5)
b2  = np.zeros(Ht)
xs2 = rng.standard_normal((6, Dt)); h0b = np.zeros(Ht)
lr = 0.5
L0, _ = loss_and_grads_task(xs2, h0b, Wx2, Wh2, b2)
for step in range(50):
    L, grads = loss_and_grads_task(xs2, h0b, Wx2, Wh2, b2)
    grads, gn = clip_grads_by_norm(grads, max_norm=5.0)
    Wx2 -= lr * grads[0]; Wh2 -= lr * grads[1]; b2 -= lr * grads[2]
L_final, _ = loss_and_grads_task(xs2, h0b, Wx2, Wh2, b2)
print(f'损失: {L0:.5f} -> {L_final:.5f} (50 步带裁剪的梯度下降)')
assert L_final < L0 * 0.1, '训练应显著降低损失'
print('✅ 前向+BPTT+裁剪+更新 自洽：损失下降，整条链路正确')

---
# ✏️ 练习

每题先写骨架(`raise NotImplementedError`)，再跑紧随其后的自测 cell。卡住可看文末 📖 参考答案。

## ✏️ 练习 1：带每步输出的 RNN 前向

扩展 `rnn_forward`：除隐藏态外，每步再经一个输出层 $y_t = W_y h_t + b_y$（用于序列标注）。

实现 `rnn_forward_with_output(xs, h0, Wx, Wh, b, Wy, by)`，返回 `H_all:(T,H)` 与 `Y_all:(T,C)`。

In [ ]:
def rnn_forward_with_output(xs, h0, Wx, Wh, b, Wy, by):
    # TODO: 逐步算 h_t（复用 rnn_cell_forward 的公式），再算 y_t = Wy@h_t + by
    #       返回 H_all:(T,H), Y_all:(T,C)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
C = 2
Wy = rng.standard_normal((C, H)) * 0.5; by = rng.standard_normal(C) * 0.1
H_all_ex, Y_all = rnn_forward_with_output(xs, np.zeros(H), Wx, Wh, b, Wy, by)
assert H_all_ex.shape == (T, H) and Y_all.shape == (T, C)
# 隐藏态应与不带输出的 rnn_forward 一致
H_ref, _ = rnn_forward(xs, np.zeros(H), Wx, Wh, b)
assert np.allclose(H_all_ex, H_ref, atol=1e-12)
# 输出层手算核对第 0 步
assert np.allclose(Y_all[0], Wy @ H_ref[0] + by, atol=1e-12)
print('✅ 练习 1 通过：每步输出 y_t = Wy h_t + by 正确')

## ✏️ 练习 2：BPTT 求 dWh（对拍数值梯度）

**只**实现对 $W_h$ 的梯度（损失同第 3 节 $L=\tfrac12\sum_t\|h_t\|^2$），体会「沿时间累加」。

实现 `grad_Wh(xs, h0, Wx, Wh, b)`，返回 `dWh`。提示：和第 3 节一样反向循环，但只累加 `dWh`。

In [ ]:
def grad_Wh(xs, h0, Wx, Wh, b):
    # TODO: 前向 rnn_forward 拿 caches；反向从 T-1 到 0，
    #       dh = H_all[t] + dh_next; da = dh*(1-h^2);
    #       dWh += outer(da, h_prev); dh_next = Wh.T @ da
    #       返回 dWh
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
dWh_ex = grad_Wh(xs, h0, Wx, Wh, b)
def L_only2(W):
    H_all, _ = rnn_forward(xs, h0, Wx, W, b)
    return 0.5 * np.sum(H_all ** 2)
err = rel_error(dWh_ex, numerical_grad(L_only2, Wh.copy()))
print(f'dWh 相对误差 = {err:.2e}')
assert dWh_ex.shape == Wh.shape and err < 1e-5
print('✅ 练习 2 通过：dWh 沿时间累加正确，过数值检验')

## ✏️ 练习 3：clip-by-value vs clip-by-norm

实现**逐元素**裁剪 `clip_by_value(grads, c)`：把每个元素裁到 $[-c, c]$。

再对比：对同一个梯度，clip-by-value 通常**改变方向**，而 clip-by-norm **保持方向**。

In [ ]:
def clip_by_value(grads, c):
    # TODO: 对每个 ndarray 用 np.clip 裁到 [-c, c]，返回新 list
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
g = [np.array([10.0, 0.1, -8.0, 0.2])]
cv = clip_by_value(g, c=1.0)
assert np.allclose(cv[0], [1.0, 0.1, -1.0, 0.2]), '逐元素裁到 [-1,1]'
# 方向对比：clip-by-value 改变方向，clip-by-norm 不改
flat0 = g[0]
cn, _ = clip_grads_by_norm(g, max_norm=1.0)
cos_value = flat0 @ cv[0] / (np.linalg.norm(flat0) * np.linalg.norm(cv[0]))
cos_norm  = flat0 @ cn[0] / (np.linalg.norm(flat0) * np.linalg.norm(cn[0]))
print(f'clip-by-value 方向余弦 = {cos_value:.4f} (≠1, 方向被改)')
print(f'clip-by-norm  方向余弦 = {cos_norm:.4f} (=1, 方向保持)')
assert cos_value < 0.999 and abs(cos_norm - 1.0) < 1e-6
print('✅ 练习 3 通过：理解两种裁剪对方向的不同影响')

## ✏️ 练习 4：谱半径与消失边界

实现 `decays(W, n_steps)`：判断在线性递推下梯度是否**消失**（最后范数 < 初始范数）。

再扫一组谱半径，找出消失/爆炸的**分界点**应在 $\rho=1$ 附近。

In [ ]:
def decays(W, n_steps=30):
    # TODO: 复用 grad_norms_through_time(W, n_steps)，
    #       返回 True 当 norms[-1] < norms[0]（梯度消失），否则 False
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
results = {}
for rho in [0.3, 0.7, 0.95, 1.05, 1.3]:
    W = make_W_with_spectral_radius(8, rho, seed=7)
    results[rho] = decays(W, 30)
print('谱半径 -> 是否消失:', results)
assert results[0.3] is True and results[0.7] is True, 'rho<1 应消失'
assert results[1.3] is False, 'rho>1 应爆炸(不消失)'
print('✅ 练习 4 通过：分界点在 rho=1，与理论一致')

---
# 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def rnn_forward_with_output(xs, h0, Wx, Wh, b, Wy, by):
    T = xs.shape[0]; H = h0.shape[0]; C = by.shape[0]
    H_all = np.zeros((T, H)); Y_all = np.zeros((T, C))
    h_prev = h0
    for t in range(T):
        h, _ = rnn_cell_forward(xs[t], h_prev, Wx, Wh, b)
        H_all[t] = h
        Y_all[t] = Wy @ h + by
        h_prev = h
    return H_all, Y_all

In [ ]:
# 练习 2 参考答案
def grad_Wh(xs, h0, Wx, Wh, b):
    H_all, caches = rnn_forward(xs, h0, Wx, Wh, b)
    T = xs.shape[0]; H = h0.shape[0]
    dWh = np.zeros_like(Wh)
    dh_next = np.zeros(H)
    for t in reversed(range(T)):
        x_t, h_prev, h = caches[t]
        dh = H_all[t] + dh_next
        da = dh * (1 - h ** 2)
        dWh += np.outer(da, h_prev)       # 沿时间累加
        dh_next = Wh.T @ da
    return dWh

In [ ]:
# 练习 3 参考答案
def clip_by_value(grads, c):
    return [np.clip(g, -c, c) for g in grads]

In [ ]:
# 练习 4 参考答案
def decays(W, n_steps=30):
    norms = grad_norms_through_time(W, n_steps)
    return bool(norms[-1] < norms[0])

---
# 🧪 真实数据胶囊：字符级序列上的 RNN 前向 + BPTT

用一个**真实的短字符串**当训练序列（char-level，Karpathy 经典玩具），把本模块的前向+BPTT 接到「真任务」：

任务：给定字符预测**下一个字符**（语言模型的最小形态）。我们在一个真实单词串上构建词表、one-hot 编码、跑一次完整的前向+BPTT，并用数值梯度检验。

> 学生骨架 cell 标了 `noverify`：先自己填，再看答案。

In [ ]:
# 真实字符串 -> 词表 -> one-hot 序列（这就是 char-RNN 的数据管线）
text = 'hello world'                      # 真实短文本
chars = sorted(set(text))
vocab = {ch: i for i, ch in enumerate(chars)}
V = len(chars)
ids = np.array([vocab[c] for c in text])
X_oh = np.eye(V)[ids[:-1]]                 # 输入：每个字符的 one-hot, (T, V)
targets = ids[1:]                          # 目标：下一个字符, (T,)
print(f'文本 {text!r} | 词表大小 V={V} | 序列长度 T={len(ids)-1}')
print('词表:', vocab)
assert X_oh.shape == (len(text) - 1, V)
print('✅ 字符级数据管线就绪（真实文本 -> one-hot 序列）')

In [ ]:
def softmax(z):
    '''数值稳定 softmax（向量版），胶囊里反复用。'''
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

assert np.allclose(softmax(np.array([0.0, 0.0])), [0.5, 0.5])
print('✅ softmax 就绪')

**🧪 胶囊练习**：实现 char-RNN 的**前向 + 交叉熵损失**。

每步：$h_t=\tanh(W_x x_t + W_h h_{t-1}+b)$，$\;y_t=W_y h_t + b_y$，$\;p_t=\mathrm{softmax}(y_t)$，损失 $L=-\sum_t \log p_t[\text{target}_t]$。返回 `L` 与所有 `caches`（含 $p_t$，反向要用）。

In [ ]:
def char_rnn_forward_loss(X_oh, targets, h0, Wx, Wh, b, Wy, by):
    # TODO: 逐步 h_t = tanh(Wx@x + Wh@h_prev + b); y=Wy@h+by; p=softmax(y)
    #       L += -log(p[target_t]); 缓存 (x, h_prev, h, p, target) 供反向
    #       返回 L, caches
    raise NotImplementedError

In [ ]:
# —— 胶囊自测（前向）——
Hh = 6
g = np.random.default_rng(3)
Wx_c = g.standard_normal((Hh, V)) * 0.3
Wh_c = g.standard_normal((Hh, Hh)) * 0.3
b_c  = np.zeros(Hh)
Wy_c = g.standard_normal((V, Hh)) * 0.3
by_c = np.zeros(V)
h0_c = np.zeros(Hh)
L, caches = char_rnn_forward_loss(X_oh, targets, h0_c, Wx_c, Wh_c, b_c, Wy_c, by_c)
# 未训练时损失应接近 log(V)（均匀分布的交叉熵）
print(f'初始损失 L = {L:.4f}，均匀基线 T*log(V) = {len(targets) * np.log(V):.4f}')
assert np.isfinite(L) and L > 0
assert abs(L - len(targets) * np.log(V)) < len(targets) * 0.5
print('✅ 胶囊前向通过：char-RNN 交叉熵损失合理（接近均匀基线）')

**🧪 胶囊练习（续）**：实现对应的 **BPTT 反向**，并用数值梯度检验 `Wy`（输出层最易验证）。

提示：softmax+交叉熵的梯度 $\partial L/\partial y_t = p_t - \text{onehot}(\text{target}_t)$（干净！），其余同第 3 节 BPTT。

In [ ]:
def char_rnn_backward(caches, h0, Wx, Wh, b, Wy, by):
    # TODO: 反向。dy = p - onehot(target); dWy += outer(dy, h); dby += dy;
    #       dh = Wy.T@dy + dh_next; da = dh*(1-h^2); 累加 dWx,dWh,db;
    #       dh_next = Wh.T@da。返回 dWx,dWh,db,dWy,dby
    raise NotImplementedError

In [ ]:
# —— 胶囊自测（反向，数值检验 Wy）——
dWx_c, dWh_c, db_c, dWy_c, dby_c = char_rnn_backward(caches, h0_c, Wx_c, Wh_c, b_c, Wy_c, by_c)
def L_of_Wy(W):
    return char_rnn_forward_loss(X_oh, targets, h0_c, Wx_c, Wh_c, b_c, W, by_c)[0]
err_Wy = rel_error(dWy_c, numerical_grad(L_of_Wy, Wy_c.copy()))
def L_of_Wh(W):
    return char_rnn_forward_loss(X_oh, targets, h0_c, Wx_c, W, b_c, Wy_c, by_c)[0]
err_Wh = rel_error(dWh_c, numerical_grad(L_of_Wh, Wh_c.copy()))
print(f'dWy 相对误差 = {err_Wy:.2e} | dWh 相对误差 = {err_Wh:.2e}')
assert err_Wy < 1e-5 and err_Wh < 1e-5
print('✅ 胶囊反向通过：char-RNN 的完整 BPTT 数值检验过关')

In [ ]:
# 📖 胶囊参考答案
def char_rnn_forward_loss(X_oh, targets, h0, Wx, Wh, b, Wy, by):
    T = X_oh.shape[0]
    h_prev = h0; L = 0.0; caches = []
    for t in range(T):
        x = X_oh[t]
        h = np.tanh(Wx @ x + Wh @ h_prev + b)
        y = Wy @ h + by
        p = softmax(y)
        L += -np.log(p[targets[t]] + 1e-12)
        caches.append((x, h_prev, h, p, targets[t]))
        h_prev = h
    return L, caches

def char_rnn_backward(caches, h0, Wx, Wh, b, Wy, by):
    H = h0.shape[0]; V = by.shape[0]
    dWx = np.zeros_like(Wx); dWh = np.zeros_like(Wh); db = np.zeros_like(b)
    dWy = np.zeros_like(Wy); dby = np.zeros_like(by)
    dh_next = np.zeros(H)
    for t in reversed(range(len(caches))):
        x, h_prev, h, p, tgt = caches[t]
        dy = p.copy(); dy[tgt] -= 1.0          # softmax+CE: p - onehot
        dWy += np.outer(dy, h); dby += dy
        dh = Wy.T @ dy + dh_next
        da = dh * (1 - h ** 2)
        dWx += np.outer(da, x); dWh += np.outer(da, h_prev); db += da
        dh_next = Wh.T @ da
    return dWx, dWh, db, dWy, dby

### 小结
- RNN = 隐藏态沿时间更新 $h_t=\tanh(W_x x_t+W_h h_{t-1}+b)$，**所有时间步共享一套权重**（时间维的参数共享）。
- 时间展开 = 深度为 T 的权重共享前馈网；BPTT = 在展开图上跑反向传播。
- **权重共享 ⇒ 梯度求和**：dWx/dWh/db 在所有时间步累加（头号 bug 来源）。
- 梯度消失/爆炸 = 循环雅可比连乘 ≈ 谱半径的指数幂；裁剪治爆炸，**消失要靠改结构**。

下一站：**模块 04 · LSTM 与 GRU** —— 用门控的加性 cell state 为梯度开一条高速公路，根治梯度消失。